# System A — Elo-PS (beginner version)

**Goal:** maintain a single MMA rating per fighter that updates after each MMA bout.

Elo answers two questions:
1) **Before** the fight, what is the win probability?
2) **After** the fight, how much should ratings move given the surprise of the result?

We extend standard Elo with two production features:
- **Tier-aware evidence** (A/B/C)
- **Method / performance shaping** (so a dominant performance moves ratings slightly more)

---

## 1) Core idea (plain English)

- If two fighters have equal rating, each is ~50% to win.
- If Fighter A's rating is higher, A is expected to win more often.
- If A wins as expected, rating changes a little.
- If A loses as an underdog, ratings change a lot.

---

## 2) The math

### 2.1 Expected win probability
Elo uses a logistic curve in rating difference:

$$
E_{A} = \frac{1}{1 + 10^{-(R_A - R_B)/s}}
$$

- $R_A, R_B$: ratings
- $s$: scale (commonly 400)

### 2.2 Update rule
$$
R_A \leftarrow R_A + K \cdot (Y - E_A)
$$
$$
R_B \leftarrow R_B + K \cdot ((1-Y) - (1-E_A))
$$

- $K$: step size (bigger means more volatile)
- $Y$: *target score* in $[0,1]$ (usually 1 win, 0 loss, 0.5 draw)

---

## 3) Tiered evidence (your A/B/C reality)

### Tier C (Tapology outcome-only)
- Use $Y = 1/0/0.5$ from the result only.
- Use smaller effective $K$ because the data is noisier.

### Tier B (UFCStats totals only)
- Use a fight-level performance score proxy to slightly shape $Y$.
- Use medium effective $K$.

### Tier A (full per-round stats)
- Compute per-round PS_round and aggregate to PS_fight.
- Use PS_fight to shape $Y$ more confidently.

**Important rule:** Draw ⇒ **no rating delta** (strict).  
NC/Overturned ⇒ **no skill update** (optional uncertainty changes happen in Glicko/System C, not here).

---

## 4) What PS shaping does (intuition)

If A wins but barely (close decision), you may want:
- small update (since it was close)

If A wins dominantly (big PS margin or early finish), you may want:
- slightly larger update (still conservative)

This shaping must be small enough to avoid turning Elo into “margin-of-victory inflation.”


In [ ]:
import sys
from datetime import date
from importlib import import_module
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

system_a_module = import_module('elo_calculator.application.ranking.system_a_elo_ps')
types_module = import_module('elo_calculator.application.ranking.types')
EloPerformanceRatingSystem = system_a_module.EloPerformanceRatingSystem
new_elo_state = system_a_module.new_elo_state
BoutEvidence = types_module.BoutEvidence
BoutOutcome = types_module.BoutOutcome
EvidenceTier = types_module.EvidenceTier
FinishMethod = types_module.FinishMethod

## 5) Worked toy example (three fights)

We'll simulate one fighter against three opponents under different tiers:
- Tier C (outcome-only)
- Tier B (totals-only with a PS_fight estimate)
- Tier A (round-based PS_fight)

The point is to show *how* the update changes, not to model real UFC outcomes.


In [ ]:
system = EloPerformanceRatingSystem()
fighter = new_elo_state('fighter')
opponents = {'opp_1': new_elo_state('opp_1'), 'opp_2': new_elo_state('opp_2'), 'opp_3': new_elo_state('opp_3')}

scenarios = [
    (
        'opp_1',
        BoutEvidence(
            bout_id='demo_1',
            event_date=date(2024, 1, 15),
            sport_key='mma',
            outcome_for_a=BoutOutcome.WIN,
            tier=EvidenceTier.C,
            method=FinishMethod.TKO,
            scheduled_rounds=3,
            ps_margin_a=0.10,
        ),
    ),
    (
        'opp_2',
        BoutEvidence(
            bout_id='demo_2',
            event_date=date(2024, 4, 20),
            sport_key='mma',
            outcome_for_a=BoutOutcome.LOSS,
            tier=EvidenceTier.B,
            method=FinishMethod.DEC,
            scheduled_rounds=3,
            ps_margin_a=-0.03,
        ),
    ),
    (
        'opp_3',
        BoutEvidence(
            bout_id='demo_3',
            event_date=date(2024, 8, 10),
            sport_key='mma',
            outcome_for_a=BoutOutcome.WIN,
            tier=EvidenceTier.A,
            method=FinishMethod.DEC,
            scheduled_rounds=5,
            is_title_fight=True,
            ps_margin_a=0.28,
        ),
    ),
]

rows = []
for fight_order, (opponent_id, evidence) in enumerate(scenarios, start=1):
    opponent_state = opponents[opponent_id]
    fighter, opponent_post, fighter_delta, _opponent_delta = system.update_bout(fighter, opponent_state, evidence)
    opponents[opponent_id] = opponent_post

    rows.append(
        {
            'fight_order': fight_order,
            'opponent_id': opponent_id,
            'outcome': evidence.outcome_for_a.value,
            'tier': evidence.tier.value,
            'method': evidence.method.value,
            'ps_margin_a': evidence.ps_margin_a,
            'expected_win_prob': round(fighter_delta.expected_win_prob, 4),
            'target_score': round(fighter_delta.target_score, 4),
            'k_effective': round(fighter_delta.k_effective, 4),
            'delta_rating': round(fighter_delta.delta_rating, 4),
            'post_rating': round(fighter_delta.post_rating, 4),
        }
    )

pd.DataFrame(rows)

## 6) What to take away

- Elo is fundamentally: **surprise-based updating**.
- Tier A/B/C mostly changes:
  - whether you can use PS to shape the target $Y$,
  - and whether you reduce effective $K$ for noisy evidence.

If you want cross-sport effects, Elo is not the right place to *directly* update MMA ratings from kickboxing bouts.
Instead, either:
- inject cross-sport priors into Elo, or
- use System C as the primary model.
